In [2]:
# 1、导包
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from datasets import load_dataset

# 2、准备数据：把原始数据，转换成SFTTrainer所需要的数据类型，
# 2.1 加载数据
datasets_dict = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0601 14:25:56.511000 38992 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [5]:
datasets_dict["train"][0]

{'conversation_id': 501,
 'category': 'dialogue',
 'conversation': [{'human': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'assistant': '高氟铍矿;配料;熔炼;回收率;脱氟率'}],
 'dataset': 'psychology'}

In [3]:
# 2.2 使用map方法，对数据进行处理，处理成conversation格式
from typing import Dict, List
def map(examples:Dict[str,List]):
    """
    对数据集进行处理
    输入：examples:原始数据，包含conversation_id，category等key
    输出：处理好的conversion数据，键是messages，值是一个列表，里面是两个json,分别表示human message 和 assistant message
    """
    conversations:List = examples["conversation"]
    new_conversations = []
    for conversation in conversations:
        # 第一层，conversation，是多个样本
        message_list = []
        for message in conversation:
            # 第二层，是一条数据当中的多个message
            for key, value in message.items():
                if key == "human":
                    message_list.append({"role":"user","content":value})
                else:
                    message_list.append({"role":"assistant","content":value})
        new_conversations.append(message_list)


            
    return {"messages":new_conversations}

mapped_datasets_dict = datasets_dict.map(function=map,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])

In [7]:
mapped_datasets_dict["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

In [ ]:
# 3、构造SFTConfig实例
config = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    #
    # max_steps=10
    logging_strategy="steps",
    logging_steps=100,
    report_to="tensorboard",
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # 评估和保存相关参数：
    eval_strategy="steps",
    eval_steps=200,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    output_dir="finetuned/Qwen3-0.6B-SFT",

    # 优化相关
    bf16=True,
    gradient_checkpointing=True,
    activation_offloading=True,
    max_length=500,
    
    # use_liger_kernel=True,
    # model_init_kwargs={"attn_implementation": "kernels-community/flash-attn2"}

    assistant_only_loss=True,
    # chat_template_path=

)

# 4、SFTTrainer实例
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B/")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B")
trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=mapped_datasets_dict["train"],
    eval_dataset=mapped_datasets_dict["test"],
    processing_class=tokenizer
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1366.70it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Truncating eval dataset: 100%|██████████| 500/500 [00:00<00:00, 41836.77 examples/s]


In [9]:
train_dataloader = trainer.get_train_dataloader()
for batch in train_dataloader:
    # print(batch)
    decoded_data = tokenizer.decode(batch["input_ids"])
    print(decoded_data[0])
    break

<|im_start|>user
关键词抽取：
以手动换挡机构疲劳寿命试验为目的,构建了一种模拟驾驶员进行选档、换挡操作的试验平台,该平台集成二自由度运动滑台与气动加载装置为一体形成换挡运动加载机构.以该机构为研究对象,通过建立换挡与选挡的运动轨迹模型,分析换挡运动加载机构位移输出与运动轨迹之间的关系,通过分析换挡机构操纵杆受力情况,分别对换挡动作和选档动作进行力学分析,并得出加载力的计算方法.最后结合电气控制技术与气动控制技术,对系统进行了试验,结果表明,系统具有可行性与正确性.<|im_end|>
<|im_start|>assistant

<think>

</think>

换挡机构;试验平台;疲劳性能<|im_end|>




In [10]:
trainer.train()
trainer.save_model(output_dir="finetuned/Qwen3-0.6B-SFT")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
[RANK 0] curr_pct=69.3% > self.virtual_memory_safe_pct=60% of virtual memory used


Step,Training Loss,Validation Loss


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [11]:
import pandas as pd

In [25]:
# 2.2 使用map方法，对数据进行处理，处理成conversation格式
from typing import Dict, List
def get_len_map(examples:Dict[str,List]):
    """
    对数据集进行处理
    输入：examples:原始数据，包含conversation_id，category等key
    输出：处理好的conversion数据，键是messages，值是一个列表，里面是两个json,分别表示human message 和 assistant message
    """
    conversations:List = examples["messages"]
    seq_len_list = []
    for conversation in conversations:
        # 第一层，conversation，是多个样本
        res = tokenizer.apply_chat_template(conversation,tokenize=True,)
        seq_len = len(res["input_ids"])
        seq_len_list.append(seq_len)



            
    return {"length":seq_len_list}

mapped_datasets_dict_with_seq_len = mapped_datasets_dict.map(function=get_len_map,batched=True,remove_columns=["messages"])

In [28]:
list_res=mapped_datasets_dict_with_seq_len["train"].to_list()

In [29]:
list_res

[{'length': 139},
 {'length': 530},
 {'length': 118},
 {'length': 173},
 {'length': 108},
 {'length': 104},
 {'length': 189},
 {'length': 340},
 {'length': 317},
 {'length': 110},
 {'length': 130},
 {'length': 152},
 {'length': 174},
 {'length': 191},
 {'length': 225},
 {'length': 172},
 {'length': 110},
 {'length': 173},
 {'length': 160},
 {'length': 232},
 {'length': 222},
 {'length': 147},
 {'length': 97},
 {'length': 278},
 {'length': 68},
 {'length': 142},
 {'length': 232},
 {'length': 165},
 {'length': 172},
 {'length': 69},
 {'length': 162},
 {'length': 154},
 {'length': 209},
 {'length': 229},
 {'length': 222},
 {'length': 109},
 {'length': 138},
 {'length': 231},
 {'length': 306},
 {'length': 158},
 {'length': 174},
 {'length': 151},
 {'length': 91},
 {'length': 165},
 {'length': 103},
 {'length': 267},
 {'length': 156},
 {'length': 165},
 {'length': 70},
 {'length': 141},
 {'length': 349},
 {'length': 227},
 {'length': 227},
 {'length': 228},
 {'length': 182},
 {'length': 379

In [30]:
all_length = [seq["length"] for seq in list_res]

In [31]:
all_length

[139,
 530,
 118,
 173,
 108,
 104,
 189,
 340,
 317,
 110,
 130,
 152,
 174,
 191,
 225,
 172,
 110,
 173,
 160,
 232,
 222,
 147,
 97,
 278,
 68,
 142,
 232,
 165,
 172,
 69,
 162,
 154,
 209,
 229,
 222,
 109,
 138,
 231,
 306,
 158,
 174,
 151,
 91,
 165,
 103,
 267,
 156,
 165,
 70,
 141,
 349,
 227,
 227,
 228,
 182,
 379,
 213,
 139,
 117,
 259,
 111,
 143,
 325,
 291,
 244,
 179,
 377,
 170,
 140,
 138,
 276,
 202,
 132,
 119,
 430,
 237,
 92,
 212,
 174,
 159,
 279,
 193,
 113,
 292,
 261,
 164,
 175,
 85,
 124,
 130,
 148,
 307,
 121,
 209,
 135,
 166,
 135,
 148,
 140,
 139,
 112,
 350,
 152,
 137,
 95,
 162,
 111,
 249,
 365,
 191,
 106,
 199,
 95,
 110,
 129,
 149,
 119,
 66,
 116,
 162,
 123,
 193,
 119,
 123,
 127,
 121,
 157,
 348,
 300,
 85,
 211,
 278,
 160,
 94,
 97,
 243,
 83,
 134,
 258,
 89,
 272,
 157,
 213,
 102,
 243,
 249,
 211,
 138,
 129,
 131,
 151,
 373,
 112,
 91,
 125,
 252,
 296,
 164,
 174,
 149,
 407,
 90,
 272,
 124,
 136,
 177,
 171,
 84,
 299,
 165

In [23]:
import pandas as pd
res_pd = pd.Series(all_length)

In [35]:

res_pd.quantile(0.9999)

np.float64(992.8016000000061)